# Diagnóstico inicial de calidad de datos
**Proyecto:** Barreras para la actividad física – PulseUp
**Evidencia:** GA1-220501094-01-AA1-EV05 · **Autor:** Santiago Varela Peña · ADSO, Ficha 3223873

**Pregunta analítica:** ¿Qué barreras impiden a los adultos realizar actividad física de forma regular, qué factores
sociodemográficos y de salud se asocian con la inactividad, y cuáles de esas barreras puede abordar una aplicación móvil
de hábitos saludables como PulseUp?

**Objetivo de este notebook:** cargar, inspeccionar y diagnosticar la calidad de los tres datasets del proyecto
antes de limpiarlos o analizarlos. En esta etapa **no se modifican los datos**: solo se describen y se documentan los problemas.

## Momento 1. Reconocimiento de los datasets

| # | Dataset | Fuente | Formato | Relación con la pregunta |
|---|---|---|---|---|
| 1 | BRFSS 2023 | CDC de EE. UU. | SAS Transport (`.XPT`) | Factores sociodemográficos y de salud asociados a la inactividad |
| 2 | Actividad física en Bogotá D.C. | Secretaría Distrital de Salud (Datos Abiertos Colombia) | CSV | Contexto local de la inactividad |
| 3 | Eurobarómetro 97.3 (ZA7888) | Comisión Europea / GESIS | Stata (`.dta`) o SPSS (`.sav`) | Barreras declaradas para no hacer actividad física |

**Aspectos de calidad a revisar:** valores faltantes, tipos de datos, códigos especiales
(no sabe / no responde), valores fuera de rango, duplicados y limitaciones para responder la pregunta.

## Momento 2. Preparación y carga de datos

In [ ]:
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*select_dtypes.*")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# En Google Colab los datos se leen desde Google Drive; en local, desde data/raw/
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_RAW = Path("/content/drive/MyDrive/pulseup-datasets")
except ModuleNotFoundError:
    DATA_RAW = Path("..") / "data" / "raw"

print("Pandas:", pd.__version__)
print("Datos en:", DATA_RAW)
print("Archivos:", sorted(p.name for p in DATA_RAW.iterdir() if p.is_file()))

In [ ]:
def buscar_archivo(*patrones):
    """Devuelve el primer archivo de la carpeta de datos que coincida con alguno de los patrones."""
    for patron in patrones:
        encontrados = sorted(DATA_RAW.glob(patron))
        if encontrados:
            return encontrados[0]
    raise FileNotFoundError(
        f"No se encontró ningún archivo con {patrones} en {DATA_RAW.resolve()}. "
        "Revisa data/raw/README.md para descargarlo."
    )

### Dataset 1: BRFSS 2023
El archivo completo tiene más de 400 000 filas y más de 300 columnas. Para no saturar la memoria,
se lee por bloques y se conservan solo las variables relacionadas con la pregunta analítica.

In [ ]:
VARIABLES_BRFSS = [
    "_STATE",     # estado de residencia
    "SEXVAR",     # sexo
    "_AGEG5YR",   # grupo de edad (5 años)
    "_AGE80",     # edad imputada (máx. 80)
    "EDUCA",      # nivel educativo
    "EMPLOY1",    # situación laboral
    "INCOME3",    # ingresos del hogar
    "EXERANY2",   # actividad física en los últimos 30 días
    "_TOTINDA",   # indicador calculado de actividad física
    "GENHLTH",    # salud general percibida
    "SLEPTIM1",   # horas de sueño
    "_BMI5",      # índice de masa corporal (2 decimales implícitos)
    "ADDEPEV3",   # diagnóstico de trastorno depresivo
    "_LLCPWT",    # peso muestral
]

ruta_brfss = buscar_archivo("LLCP2023*", "*.XPT*", "*.xpt")
bloques = []
with pd.read_sas(ruta_brfss, format="xport", chunksize=50_000, encoding="latin-1") as lector:
    for bloque in lector:
        columnas = [c for c in VARIABLES_BRFSS if c in bloque.columns]
        bloques.append(bloque[columnas])
df_brfss = pd.concat(bloques, ignore_index=True)

faltan = [c for c in VARIABLES_BRFSS if c not in df_brfss.columns]
print(f"BRFSS cargado: {df_brfss.shape[0]:,} filas x {df_brfss.shape[1]} columnas")
print("Variables no encontradas en el archivo:", faltan or "ninguna")

### Dataset 2: Actividad física en Bogotá D.C.

In [ ]:
ruta_bogota = buscar_archivo("*actividad*fisica*.csv", "*.csv", "*.xlsx")
if ruta_bogota.suffix == ".xlsx":
    df_bogota = pd.read_excel(ruta_bogota)
else:
    df_bogota = pd.read_csv(ruta_bogota, encoding="utf-8")
print(f"Bogotá cargado: {df_bogota.shape[0]:,} filas x {df_bogota.shape[1]} columnas")

### Dataset 3: Eurobarómetro 97.3 (ZA7888)
Se intenta conservar las etiquetas de las respuestas. Si el archivo trae etiquetas duplicadas,
se carga con los códigos numéricos.

**Nota:** este dataset requiere una cuenta de GESIS y no se puede descargar automáticamente. Si el archivo aún no está en `data/raw/`, el notebook continúa con los otros dos datasets y las celdas del Eurobarómetro muestran un aviso en lugar de fallar.

In [ ]:
try:
    ruta_euro = buscar_archivo("ZA7888*.dta", "ZA7888*.sav")
except FileNotFoundError as e:
    ruta_euro = None
    df_eurobarometro = None
    print("AVISO:", e)

if ruta_euro is not None:
    if ruta_euro.suffix == ".dta":
        try:
            df_eurobarometro = pd.read_stata(ruta_euro)
            etiquetas = True
        except ValueError:
            df_eurobarometro = pd.read_stata(ruta_euro, convert_categoricals=False)
            etiquetas = False
    else:
        df_eurobarometro = pd.read_spss(ruta_euro)
        etiquetas = True
    print(f"Eurobarómetro cargado: {df_eurobarometro.shape[0]:,} filas x {df_eurobarometro.shape[1]} columnas")
    print("Etiquetas de respuesta conservadas:", etiquetas)

## Momento 3. Inspección de la estructura

### Dataset 1: BRFSS 2023

In [ ]:
df_brfss.head()

In [ ]:
df_brfss.tail()

In [ ]:
df_brfss.shape

In [ ]:
df_brfss.columns

In [ ]:
df_brfss.dtypes

In [ ]:
df_brfss.info()

In [ ]:
df_brfss.describe()

In [ ]:
if df_brfss.select_dtypes(include=["object", "category"]).shape[1] > 0:
    display(df_brfss.describe(include="object") if df_brfss.select_dtypes(include="object").shape[1] else df_brfss.describe(include="category"))
else:
    print("Este dataset no tiene columnas de tipo object.")

**Interpretación:** _(completar con lo observado en las salidas anteriores)_

### Dataset 2: Bogotá

In [ ]:
df_bogota.head()

In [ ]:
df_bogota.tail()

In [ ]:
df_bogota.shape

In [ ]:
df_bogota.columns

In [ ]:
df_bogota.dtypes

In [ ]:
df_bogota.info()

In [ ]:
df_bogota.describe()

In [ ]:
if df_bogota.select_dtypes(include=["object", "category"]).shape[1] > 0:
    display(df_bogota.describe(include="object") if df_bogota.select_dtypes(include="object").shape[1] else df_bogota.describe(include="category"))
else:
    print("Este dataset no tiene columnas de tipo object.")

**Interpretación:** _(completar con lo observado en las salidas anteriores)_

### Dataset 3: Eurobarómetro 97.3

In [ ]:
if df_eurobarometro is not None:
    display(df_eurobarometro.head())
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

In [ ]:
if df_eurobarometro is not None:
    display(df_eurobarometro.tail())
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

In [ ]:
if df_eurobarometro is not None:
    print(df_eurobarometro.shape)
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

In [ ]:
if df_eurobarometro is not None:
    print(df_eurobarometro.columns)
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

In [ ]:
if df_eurobarometro is not None:
    display(df_eurobarometro.dtypes)
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

In [ ]:
if df_eurobarometro is not None:
    df_eurobarometro.info()
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

In [ ]:
if df_eurobarometro is not None:
    display(df_eurobarometro.describe())
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

In [ ]:
if df_eurobarometro is not None:
    if df_eurobarometro.select_dtypes(include=["object", "category"]).shape[1] > 0:
        display(df_eurobarometro.describe(include="object") if df_eurobarometro.select_dtypes(include="object").shape[1] else df_eurobarometro.describe(include="category"))
    else:
        print("Este dataset no tiene columnas de tipo object.")
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

**Interpretación:** _(completar con lo observado en las salidas anteriores)_

### Tabla de diagnóstico estructural

In [ ]:
def resumen_tipos(df):
    conteo = df.dtypes.astype(str).value_counts()
    return ", ".join(f"{tipo}: {n}" for tipo, n in conteo.items())

tabla_estructural = pd.DataFrame([
    {"Dataset": nombre, "Filas": df.shape[0], "Columnas": df.shape[1], "Tipos de datos": resumen_tipos(df)}
    for nombre, df in [("BRFSS 2023", df_brfss), ("Bogotá", df_bogota), ("Eurobarómetro 97.3", df_eurobarometro)]
    if df is not None
])
tabla_estructural

## Momento 4. Diagnóstico de calidad

In [ ]:
def resumen_faltantes(df):
    """Columnas con valores faltantes, ordenadas de mayor a menor."""
    faltantes = df.isna().sum()
    tabla = pd.DataFrame({"faltantes": faltantes, "porcentaje": (faltantes / len(df) * 100).round(2)})
    return tabla[tabla["faltantes"] > 0].sort_values("faltantes", ascending=False)


def texto_que_parece_numero(df):
    """Columnas de texto en las que al menos la mitad de los valores son números escritos como texto."""
    sospechosas = {}
    for col in df.select_dtypes(include=["object"]).columns:
        valores = df[col].dropna().astype(str).str.replace(",", ".", regex=False).str.strip()
        proporcion = pd.to_numeric(valores, errors="coerce").notna().mean() if len(valores) else 0
        if proporcion >= 0.5:
            sospechosas[col] = f"{proporcion:.0%} numéricos"
    return sospechosas

### Dataset 1: BRFSS 2023

In [ ]:
print("1. Valores faltantes")
display(resumen_faltantes(df_brfss))

print("2. Tipos de datos")
print(df_brfss.dtypes.value_counts())

print("3. Filas duplicadas:", df_brfss.duplicated().sum())

In [ ]:
# 4. Códigos especiales: en el BRFSS, 7/9 (o 77/99, 777/999) significan "no sabe" o "no responde".
#    Pandas los trata como números válidos, así que pueden distorsionar promedios.
codigos_especiales = {
    "EXERANY2": [7, 9], "GENHLTH": [7, 9], "ADDEPEV3": [7, 9], "EDUCA": [9],
    "EMPLOY1": [9], "INCOME3": [77, 99], "SLEPTIM1": [77, 99], "_TOTINDA": [9], "_AGEG5YR": [14],
}
filas = []
for col, codigos in codigos_especiales.items():
    if col in df_brfss.columns:
        n = df_brfss[col].isin(codigos).sum()
        filas.append({"variable": col, "códigos": codigos, "registros": n, "porcentaje": round(n / len(df_brfss) * 100, 2)})
pd.DataFrame(filas)

In [ ]:
# 5. Rangos sospechosos
print("SLEPTIM1 (horas de sueño) mayores a 24 sin contar 77/99:",
      ((df_brfss["SLEPTIM1"] > 24) & ~df_brfss["SLEPTIM1"].isin([77, 99])).sum())
print("_BMI5 tiene 2 decimales implícitos (ej. 2534 = 25.34). Rango actual:",
      df_brfss["_BMI5"].min(), "-", df_brfss["_BMI5"].max())
print("Valores únicos de EXERANY2:", sorted(df_brfss["EXERANY2"].dropna().unique()))

### Dataset 2: Bogotá

In [ ]:
print("1. Valores faltantes")
display(resumen_faltantes(df_bogota))

print("2. Columnas de texto que parecen números:", texto_que_parece_numero(df_bogota) or "ninguna")

print("3. Filas duplicadas:", df_bogota.duplicated().sum())

# 4. Porcentajes fuera del rango 0-100 (solo columnas que representan porcentajes o proporciones)
cols_pct = [c for c in df_bogota.columns if any(p in c.lower() for p in ("porc", "propor", "%"))]
for col in cols_pct:
    valores = pd.to_numeric(df_bogota[col].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    fuera = ((valores < 0) | (valores > 100)).sum()
    print(f"   {col}: {fuera} valores fuera de 0-100; {valores.isna().sum()} no convertibles a número")
if not cols_pct:
    print("   No se identificaron columnas de porcentaje por su nombre; revisar manualmente.")

In [ ]:
# Valores únicos de las columnas de texto (para detectar categorías mal escritas)
for col in df_bogota.select_dtypes(include=["object"]).columns:
    print(col, "->", df_bogota[col].nunique(), "valores únicos")
    print("   ", df_bogota[col].dropna().unique()[:15])

### Dataset 3: Eurobarómetro 97.3

In [ ]:
if df_eurobarometro is not None:
    print("1. Valores faltantes (20 columnas con más ausentes)")
    display(resumen_faltantes(df_eurobarometro).head(20))
    print("Columnas con faltantes:", (df_eurobarometro.isna().sum() > 0).sum(), "de", df_eurobarometro.shape[1])

    print("2. Tipos de datos")
    print(df_eurobarometro.dtypes.astype(str).value_counts())

    print("3. Filas duplicadas:", df_eurobarometro.duplicated().sum())
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

In [ ]:
if df_eurobarometro is not None:
    # 4. Respuestas tipo "no sabe" / "no responde" en las variables categóricas
    patron = r"(?i)don.t know|\bdk\b|refus|inap|not asked"
    categoricas = df_eurobarometro.select_dtypes(include=["category", "object"]).columns
    conteo = {}
    for col in categoricas:
        n = df_eurobarometro[col].astype(str).str.contains(patron, regex=True, na=False).sum()
        if n:
            conteo[col] = n
    print(f"Variables con respuestas 'no sabe/no responde/no aplica': {len(conteo)}")
    display(pd.Series(conteo, name="registros").sort_values(ascending=False).head(20))
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

### Variables del módulo de deporte del Eurobarómetro
El Eurobarómetro tiene cientos de variables. Esta celda busca, en las etiquetas de las variables, las preguntas
sobre deporte y actividad física (frecuencia, lugar, motivos y **barreras**), que son las que se usarán en la preparación.

In [ ]:
if ruta_euro is not None:
    if ruta_euro.suffix == ".dta":
        with pd.io.stata.StataReader(ruta_euro) as lector:
            etiquetas_vars = lector.variable_labels()
    else:
        import pyreadstat
        _, meta = pyreadstat.read_sav(ruta_euro, metadataonly=True)
        etiquetas_vars = dict(zip(meta.column_names, meta.column_labels))

    patron_deporte = r"(?i)sport|exercis|physical|activ|reason|prevent|motiv|cycl|walk"
    modulo = pd.Series(etiquetas_vars, name="etiqueta").str.strip()
    modulo = modulo[modulo.str.contains(patron_deporte, regex=True, na=False)]
    print(f"Variables relacionadas con deporte y actividad física: {len(modulo)}")
    with pd.option_context("display.max_rows", 300, "display.max_colwidth", 120):
        display(modulo.to_frame())

    # Variables sociodemográficas útiles para el análisis
    patron_socio = r"(?i)country|nation|gender|sex|age|education|occupation|weight"
    socio = pd.Series(etiquetas_vars, name="etiqueta").str.strip()
    with pd.option_context("display.max_rows", 100, "display.max_colwidth", 120):
        display(socio[socio.str.contains(patron_socio, regex=True, na=False)].head(40).to_frame())
else:
    print("Eurobarómetro no disponible.")

## Tabla de hallazgos de calidad
_(Se completa con los resultados obtenidos arriba.)_

| Dataset | Problema identificado | Evidencia encontrada | Posible impacto | Acción propuesta |
|---|---|---|---|---|
| | | | | |

## Conclusiones
_(Se completan con los resultados obtenidos.)_